# 02 — Benchmark: 4-arm GraphRAG vs PlainRAG ablation

Thin Colab wrapper around `scripts/run_benchmark.py` and `scripts/compare.py`.

**Colab Pro:** Runtime → Change runtime type → **A100 GPU** + **High-RAM**. This run is
LLM-inference-bound (~800 generations from a reasoning model), so the faster GPU is what
cuts wall-clock. Add `ARANGO_PASS` in the Colab **Secrets** panel. Run `01_ingest.ipynb` first.

Arms (each isolates one component):

| arm | adds |
| --- | --- |
| `plain` | vector top-k chunks (baseline) |
| `plain_rr` | + cross-encoder reranker |
| `graph` | + parent-paper expansion (full abstracts) |
| `graph_concepts` | + MeSH concept-hop expansion |

The runner retries failed questions and auto-restarts Ollama if it crashes, and it
checkpoints every 25 questions — so a transient 500 can't abort an arm.

In [ ]:
# Confirm the GPU. A100 is ideal; T4/L4 also work (slower).
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [15]:
# A) Clean reset → one current copy (removes the doubled folder)
%cd /content
!rm -rf Knowledge_Graph_Question_Answering
!git clone -b revamp https://github.com/vardhjain/Knowledge_Graph_Question_Answering.git -q
%cd Knowledge_Graph_Question_Answering
!pip install -q -r requirements.txt

/content
/content/Knowledge_Graph_Question_Answering


In [16]:
# B) Env — full context, bounded generation (corpus re-downloads once this run)
import os
from google.colab import userdata
os.environ['ARANGO_PASS'] = userdata.get('ARANGO_PASS')
os.environ['LLM_NUM_CTX'] = '8192'
os.environ['LLM_NUM_PREDICT'] = '1024'
print('ARANGO_PASS set:', bool(os.environ.get('ARANGO_PASS')))

ARANGO_PASS set: True


In [17]:
# C) Ollama + model (idempotent; fast if already pulled this session)
!which ollama || (apt-get install -y zstd -q && curl -fsSL https://ollama.com/install.sh | sh)
import subprocess, time
subprocess.Popen(['ollama','serve']); time.sleep(5)
!ollama pull deepseek-r1:8b

/usr/local/bin/ollama



In [ ]:
# D) Full run — all four arms (~3 hrs on the 80GB A100)
for arm in ['plain', 'plain_rr', 'graph', 'graph_concepts']:
    print(f'\n===== {arm} =====')
    !python scripts/run_benchmark.py --arm {arm} --n 200


===== plain =====
[Ollama] Ensuring server is healthy...
[ArangoDB] Connected.
[Corpus] Loading chunk store from ArangoDB (cached after first run)...


In [ ]:
# E) Aggregate → table, McNemar tests, figure
!python scripts/compare.py
from IPython.display import Image, display, Markdown
display(Markdown(open('results/summary.md').read()))
display(Image('results/ablation.png'))